# Joining Land Registry Price Data with Energy Performance Certificates (2015 - 2025)

#### Load Price data

In [141]:
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.expand_frame_repr', False)


folder = Path('../data/raw')
file_gen = folder.glob('ppd_data*.csv')
pp_raw = pd.concat([pd.read_csv(f) for f in file_gen], ignore_index=True)
pp_raw['deed_date'] = pp_raw['deed_date'].astype('datetime64[ns]')
pp_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 135557 entries, 0 to 135556
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   unique_id             135557 non-null  str           
 1   price_paid            135557 non-null  int64         
 2   deed_date             135557 non-null  datetime64[ns]
 3   postcode              135115 non-null  str           
 4   property_type         135557 non-null  str           
 5   new_build             135557 non-null  str           
 6   estate_type           135557 non-null  str           
 7   saon                  11820 non-null   str           
 8   paon                  135557 non-null  str           
 9   street                132302 non-null  str           
 10  locality              57572 non-null   str           
 11  town                  135557 non-null  str           
 12  district              135557 non-null  str           
 13  county    

#### Load EPC data

In [142]:
epc_raw = pd.read_csv('../data/raw/EDC_OXFORDSHIRE_2015_2025.csv', usecols= [
    'certificate_number', 'address1', 'address2', 'address3', 'postcode',
    'posttown', 'address', 'local_authority', 'local_authority_label',
    'built_form', 'construction_age_band', 'current_energy_efficiency',
    'current_energy_rating', 'energy_consumption_current', 'extension_count', 'floor_height', 'heating_cost_current',
    'inspection_date', 'lodgement_date', 'number_habitable_rooms',
    'number_heated_rooms', 'potential_energy_efficiency', 'property_type',
    'tenure', 'total_floor_area', 'transaction_type', 'uprn', 'uprn_source'
])
epc_raw['inspection_date'] = epc_raw['inspection_date'].astype('datetime64[ns]')
epc_raw['lodgement_date'] = epc_raw['lodgement_date'].astype('datetime64[ns]')
epc_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 216054 entries, 0 to 216053
Data columns (total 28 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   certificate_number           216054 non-null  str           
 1   address1                     216054 non-null  str           
 2   address2                     117532 non-null  str           
 3   address3                     27304 non-null   str           
 4   postcode                     216054 non-null  str           
 5   posttown                     213327 non-null  str           
 6   address                      216054 non-null  str           
 7   local_authority              216054 non-null  str           
 8   local_authority_label        216054 non-null  str           
 9   built_form                   213548 non-null  str           
 10  construction_age_band        216054 non-null  str           
 11  current_energy_efficiency    216054 n

In [143]:
print(pp_raw[['postcode','saon', 'paon', 'street', 'locality', 'town']].head())
print(epc_raw[['postcode', 'address3', 'address2', 'address1', 'posttown']].head())

   postcode           saon                  paon street   locality       town
0  HP18 9UH   PARK COTTAGE   CHILLING PLACE FARM    NaN      BRILL  AYLESBURY
1  HP18 9UY            NaN         OAKCROFT FARM    NaN  BOARSTALL  AYLESBURY
2  HP18 9UZ            NaN         TUTHER CORNER    NaN      BRILL  AYLESBURY
3  HP18 9UZ            NaN         TUTHER CORNER    NaN      BRILL  AYLESBURY
4  HP18 9XA  HEADLEY HOUSE  PIDDINGTON GATE FARM    NaN      BRILL  AYLESBURY
   postcode address3                address2                  address1         posttown
0   OX4 3UJ      NaN                     NaN            10 Newman Road           OXFORD
1  OX33 1NL      NaN                Wheatley           10 Barlow Close           OXFORD
2   OX4 3PG      NaN                     NaN          65 Campbell Road           OXFORD
3   OX7 6BE      NaN  Shipton-Under-Wychwood     12 St. Michaels Close  CHIPPING NORTON
4  OX16 5BG      NaN            Tramway Road  Flat 10, Canalside House          BANBURY


In [144]:
counts = pp_raw['postcode'].value_counts()
print(counts[counts == 10])

postcode
MK18 4AS    10
OX15 0PT    10
OX15 0QT    10
OX15 0RF    10
OX15 0RU    10
            ..
OX7 6DY     10
OX7 6DZ     10
OX7 6SY     10
OX7 7BX     10
OX7 7HD     10
Name: count, Length: 648, dtype: int64


In [145]:
print(pp_raw[pp_raw['postcode'] == 'HP18 9UH'][['postcode', 'locality', 'street', 'saon', 'paon', 'town']].head())
print(epc_raw[epc_raw['postcode'] == 'HP18 9UH'][['postcode', 'address3', 'address2', 'address1', 'posttown', 'address']].head())

   postcode locality street          saon                 paon       town
0  HP18 9UH    BRILL    NaN  PARK COTTAGE  CHILLING PLACE FARM  AYLESBURY
       postcode address3             address2                       address1   posttown                                   address
27420  HP18 9UH      NaN                Brill  Oak Lodge Chilling Place Farm  AYLESBURY      Oak Lodge Chilling Place Farm, Brill
60325  HP18 9UH    Brill  Chilling Place Farm                   Park Cottage  AYLESBURY  Park Cottage, Chilling Place Farm, Brill


In [146]:
print(pp_raw[pp_raw['postcode'] == 'OX12 9RG'][['postcode', 'locality', 'street', 'saon', 'paon', 'town']].sort_values('saon', ascending=True).head(10))
print(epc_raw[epc_raw['postcode'] == 'OX12 9RG'][['postcode', 'address3', 'address2', 'address1', 'posttown', 'address']].sort_values('address1', ascending=True).head(10))

       postcode        locality        street saon        paon     town
92129  OX12 9RG  LETCOMBE REGIS  SOUTH STREET    1  BROOK VIEW  WANTAGE
92130  OX12 9RG  LETCOMBE REGIS  SOUTH STREET   10  BROOK VIEW  WANTAGE
92131  OX12 9RG  LETCOMBE REGIS  SOUTH STREET   11  BROOK VIEW  WANTAGE
92132  OX12 9RG  LETCOMBE REGIS  SOUTH STREET   11  BROOK VIEW  WANTAGE
92133  OX12 9RG  LETCOMBE REGIS  SOUTH STREET   13  BROOK VIEW  WANTAGE
92134  OX12 9RG  LETCOMBE REGIS  SOUTH STREET   13  BROOK VIEW  WANTAGE
92135  OX12 9RG  LETCOMBE REGIS  SOUTH STREET   15  BROOK VIEW  WANTAGE
92136  OX12 9RG  LETCOMBE REGIS  SOUTH STREET   15  BROOK VIEW  WANTAGE
92137  OX12 9RG  LETCOMBE REGIS  SOUTH STREET   16  BROOK VIEW  WANTAGE
92138  OX12 9RG  LETCOMBE REGIS  SOUTH STREET   17  BROOK VIEW  WANTAGE
        postcode        address3      address2       address1 posttown                                      address
205645  OX12 9RG  Letcombe Regis  South Street   1 Brook View  WANTAGE   1 Brook View, South

- Comparison of address data
    - PP full caps vs EPC capital case
    - pp_'locality' is typically epc_'address3' but can be epc_'address2', depending on how the address is split
    - pp_'saon' or pp_'saon'+pp_'paon' can be epc_'address1'
    - pp_'town' == epc_'post_town'
    - Both appear use 'FLAT 8' vs 'Flat 8'

In [147]:
print(pp_raw[pp_raw['postcode'] == 'MK18 4AS'][['postcode', 'locality', 'street', 'saon', 'paon', 'town']].sort_values('saon', ascending=False).head(10))
print(epc_raw[epc_raw['postcode'] == 'MK18 4AS'][['postcode', 'address3', 'address2', 'address1', 'posttown', 'address']].sort_values('address1', ascending=False).head(10))

    postcode locality        street saon             paon        town
41  MK18 4AS  FINMERE  FULWELL ROAD  NaN      FIVE GABLES  BUCKINGHAM
42  MK18 4AS  FINMERE  FULWELL ROAD  NaN       HEDGELANDS  BUCKINGHAM
43  MK18 4AS  FINMERE  FULWELL ROAD  NaN        HIGH ACRE  BUCKINGHAM
44  MK18 4AS  FINMERE  FULWELL ROAD  NaN        OAK HOUSE  BUCKINGHAM
45  MK18 4AS  FINMERE  FULWELL ROAD  NaN        OAK HOUSE  BUCKINGHAM
46  MK18 4AS  FINMERE  FULWELL ROAD  NaN  STONEWELL HOUSE  BUCKINGHAM
47  MK18 4AS  FINMERE  FULWELL ROAD  NaN   SYCAMORE HOUSE  BUCKINGHAM
48  MK18 4AS  FINMERE  FULWELL ROAD  NaN   SYCAMORE HOUSE  BUCKINGHAM
49  MK18 4AS  FINMERE  FULWELL ROAD  NaN   SYCAMORE HOUSE  BUCKINGHAM
50  MK18 4AS  FINMERE  FULWELL ROAD  NaN        THE HAVEN  BUCKINGHAM
        postcode address3      address2         address1    posttown                                address
175699  MK18 4AS  Finmere  Fulwell Road      Top Gardens  BUCKINGHAM     Top Gardens, Fulwell Road, Finmere
53071   MK18 4

- Matches well on postcode and paon = address1, saon is null
- One record where posttown uses the value put in address3 for the other records

In [148]:
print(pp_raw[pp_raw['postcode'] == 'OX15 0PT'][['postcode', 'locality', 'street', 'saon', 'paon', 'town']].sort_values('paon', ascending=False).head(10))
print(epc_raw[epc_raw['postcode'] == 'OX15 0PT'][['postcode', 'address3', 'address2', 'address1', 'posttown', 'address']].sort_values('address1', ascending=False).head(10))

     postcode locality        street saon                paon     town
300  OX15 0PT  CLIFTON  CHAPEL CLOSE  NaN         THE WILLOWS  BANBURY
298  OX15 0PT  CLIFTON  CHAPEL CLOSE  NaN       ORCHARD HOUSE  BANBURY
299  OX15 0PT  CLIFTON  CHAPEL CLOSE  NaN       ORCHARD HOUSE  BANBURY
297  OX15 0PT  CLIFTON  CHAPEL CLOSE  NaN  OLD WESLYAN CHAPEL  BANBURY
296  OX15 0PT  CLIFTON  CHAPEL CLOSE  NaN           HOMELANDS  BANBURY
295  OX15 0PT  CLIFTON  CHAPEL CLOSE  NaN        HADLEY HOUSE  BANBURY
294  OX15 0PT  CLIFTON  CHAPEL CLOSE  NaN       BAMFORD HOUSE  BANBURY
293  OX15 0PT  CLIFTON  CHAPEL CLOSE  NaN       APPLE COTTAGE  BANBURY
302  OX15 0PT  CLIFTON  CHAPEL CLOSE  NaN                   5  BANBURY
301  OX15 0PT  CLIFTON  CHAPEL CLOSE  NaN                   4  BANBURY
        postcode address3      address2            address1 posttown                                    address
130406  OX15 0PT  Clifton  Chapel Close       The Homestead  BANBURY       The Homestead, Chapel Close, Cli

Fix trailing whitespace

In [149]:
pp = pp_raw.copy()

for c in pp.select_dtypes(include=['string']).columns:
    pp[c] = pp[c].str.strip()
    print(f"{c} stripped")

unique_id stripped
postcode stripped
property_type stripped
new_build stripped
estate_type stripped
saon stripped
paon stripped
street stripped
locality stripped
town stripped
district stripped
county stripped
transaction_category stripped
linked_data_uri stripped


In [150]:
epc = epc_raw.copy()

for c in epc.select_dtypes(include=['string']).columns:
    epc[c] = epc[c].str.strip()
    print(f"{c} stripped")

certificate_number stripped
address1 stripped
address2 stripped
address3 stripped
postcode stripped
posttown stripped
address stripped
local_authority stripped
local_authority_label stripped
built_form stripped
construction_age_band stripped
current_energy_rating stripped
property_type stripped
tenure stripped
transaction_type stripped
uprn_source stripped


Remove commas

In [151]:
epc['address1'] = epc['address1'].str.replace(',', '', regex=False)

- For loops over columns are fast (over rows are slow). List comprehensions wont work as they lose the index.

Generate combined addresses (space separated) and match cases

In [152]:
pp['address'] = pp['paon'].str.cat([pp['saon'], pp['street'], pp['locality'], pp['postcode']], sep=' ', na_rep='')
print(pp['address'])
epc['address'] = epc['address1'].str.cat([epc['address2'], epc['address3'], epc['postcode']], sep=' ', na_rep='').apply(str.upper)
print(epc['address'])

0          CHILLING PLACE FARM PARK COTTAGE  BRILL HP18 9UH
1                        OAKCROFT FARM   BOARSTALL HP18 9UY
2                            TUTHER CORNER   BRILL HP18 9UZ
3                            TUTHER CORNER   BRILL HP18 9UZ
4         PIDDINGTON GATE FARM HEADLEY HOUSE  BRILL HP18...
                                ...                        
135552                   THE BUTCHERS GROUND   NORTH LEIGH 
135553                  TURLEY FARM LOTS 2 3 AND 4  HAILEY 
135554                        TURLEY LANE PLOT 154  HAILEY 
135555                       PLOTS 79-84   LONG HANBOROUGH 
135556                                114 GARAGE NEW ROAD  
Name: address, Length: 135557, dtype: str
0                                  10 NEWMAN ROAD   OX4 3UJ
1                        10 BARLOW CLOSE WHEATLEY  OX33 1NL
2                                65 CAMPBELL ROAD   OX4 3PG
3         12 ST. MICHAELS CLOSE SHIPTON-UNDER-WYCHWOOD  ...
4            FLAT 10 CANALSIDE HOUSE TRAMWAY ROAD  OX16 5B

- The space separator is being added before/after null values (even though they are empty strings)
- Some commas are making it into the combined addresses

In [153]:
pp['address_tokens'] = pp['address'].str.split().apply(frozenset)
epc['address_tokens'] = epc['address'].str.split().apply(frozenset)
print(epc['address_tokens'])

0                   frozenset({3UJ, 10, ROAD, OX4, NEWMAN})
1         frozenset({WHEATLEY, BARLOW, 10, CLOSE, OX33, ...
2                 frozenset({3PG, ROAD, OX4, 65, CAMPBELL})
3         frozenset({12, 6BE, CLOSE, OX7, SHIPTON-UNDER-...
4         frozenset({10, ROAD, HOUSE, CANALSIDE, TRAMWAY...
                                ...                        
216049                frozenset({APPLETONS, 7GG, OX12, 30})
216050        frozenset({ROAD, WELL, 6ED, 1D, WALTON, OX2})
216051             frozenset({ROAD, SYCAMORE, 8, 9EJ, OX2})
216052         frozenset({SPENCER, 68, OX4, 4SW, CRESCENT})
216053    frozenset({WHEATLEY, CLOSE, OX33, 21, OCHRE, 1...
Name: address_tokens, Length: 216054, dtype: object


- Additional spaces are removed during split
- Use .apply(set) rather than set() so that the function is applied to each row of the new series, rather than trying to apply it to the entire series at once

In [154]:
first_join = pp.merge(epc, on='address_tokens', how='inner')

print(f"""New table has {first_join.shape[0]} rows, but this includes multiple EPC records per property matched. 
      Matched {first_join['unique_id'].nunique()} Price paid records with an EPC certificate. Match rate: {100* first_join['unique_id'].nunique()/pp.shape[0]:.1f}%""")


New table has 107935 rows, but this includes multiple EPC records per property matched. 
      Matched 96461 Price paid records with an EPC certificate. Match rate: 71.2%


In [155]:
print(first_join.sort_index(axis=1).sample(20))

                         address1           address2     address3                                     address_tokens                                     address_x                                     address_y     built_form        certificate_number            construction_age_band       county  current_energy_efficiency current_energy_rating  deed_date             district  energy_consumption_current estate_type  extension_count  floor_height  heating_cost_current inspection_date                                    linked_data_uri local_authority local_authority_label           locality lodgement_date new_build  number_habitable_rooms  number_heated_rooms             paon postcode_x postcode_y          posttown  potential_energy_efficiency  price_paid property_type_x property_type_y saon                  street            tenure  total_floor_area              town transaction_category   transaction_type                             unique_id          uprn      uprn_source
80218         

#### Second join

In [156]:
epc_address_cols = ['address1', 'address2', 'address3', 'postcode']
pp_address_cols = ['paon', 'saon', 'street', 'locality']

print(f"Commas replaced in epc:\n{epc[epc_address_cols].apply(lambda col: col.str.count(',')).sum()}")
print(f"Commas replaced in pp:\n{pp[pp_address_cols].apply(lambda col: col.str.count(',')).sum()}")

for c in epc_address_cols:
    epc[c] = epc[c].str.replace(',', '', regex=False)
for c in pp_address_cols:
    pp[c] = pp[c].str.replace(',', '', regex=False)

Commas replaced in epc:
address1       0.0
address2    3910.0
address3     137.0
postcode       0.0
dtype: float64
Commas replaced in pp:
paon        4356.0
saon          33.0
street         0.0
locality       0.0
dtype: float64


In [157]:
pp['address'] = pp['paon'].str.cat([pp['saon'], pp['street'], pp['locality'], pp['postcode']], sep=' ', na_rep='')
epc['address'] = epc['address1'].str.cat([epc['address2'], epc['address3'], epc['postcode']], sep=' ', na_rep='').apply(str.upper)

pp['address_tokens'] = pp['address'].str.split().apply(frozenset)
epc['address_tokens'] = epc['address'].str.split().apply(frozenset)

second_join = pp.merge(epc, on='address_tokens', how='inner')

print(f"""New table has {second_join.shape[0]} rows, but this includes multiple EPC records per property matched. 
      Matched {second_join['unique_id'].nunique()} Price paid records with an EPC certificate. Match rate: {100* second_join['unique_id'].nunique()/pp.shape[0]:.1f}%""")


New table has 110478 rows, but this includes multiple EPC records per property matched. 
      Matched 98784 Price paid records with an EPC certificate. Match rate: 72.9%


Filtering EPC matches